In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)
model

/opt/anaconda3/envs/langchain/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '0.2.17'}}, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x151edc1d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x151e0b390>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hello, I am Madhav and I am an AI engineer")])

AIMessage(content='Hello Madhav, nice to meet you. As an AI engineer, you must be working on some exciting projects. What kind of AI-related work do you specialize in, machine learning, natural language processing, computer vision, or something else?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 48, 'total_tokens': 98, 'completion_time': 0.064388524, 'completion_tokens_details': None, 'prompt_time': 0.002445301, 'prompt_tokens_details': None, 'queue_time': 0.052065548, 'total_time': 0.066833825}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f80ae-5506-7472-9f4a-b11077d227f8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 50, 'total_tokens': 98})

In [6]:
from langchain_core.messages import AIMessage
model.invoke([HumanMessage(content="Hello, I am Madhav and I am an AI engineer"),
 AIMessage(content="Hello Madhav, nice to meet you. As an AI engineer, you must be working on some exciting projects. What kind of AI-related work do you specialize in, machine learning, natural language processing, computer vision, or something else?"),
 HumanMessage(content="Hey what is my name and what do i do?")])

AIMessage(content="Your name is Madhav, and you're an AI engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 118, 'total_tokens': 133, 'completion_time': 0.017500391, 'completion_tokens_details': None, 'prompt_time': 0.009459437, 'prompt_tokens_details': None, 'queue_time': 0.04912458, 'total_time': 0.026959828}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f80b3-05d7-7f30-b690-9db8f8cbe417-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 118, 'output_tokens': 15, 'total_tokens': 133})

In [10]:
##Message History
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model,get_session_history)

/opt/anaconda3/envs/langchain/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [11]:
config = {"configurable": {"session_id": "chat_1"}}

In [12]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Madhav and I am an AI engineer")],
    config = config
)
response.content

"Nice to meet you, Madhav! As an AI engineer, that's an exciting field to be in. What kind of projects have you been working on lately? Are you interested in areas such as natural language processing, computer vision, or perhaps developing conversational AI systems like myself?"

In [13]:
config = {"configurable": {"session_id": "chat_2"}}
response = with_message_history.invoke(
    [HumanMessage(content="What is my name")],
    config = config
)
response.content

"I don't have any information about your name. I'm a large language model, I don't have the ability to retain personal information or recall previous conversations. Each time you interact with me, it's a new conversation. If you'd like to share your name with me, I'd be happy to chat with you!"

### Prompt Template

In [14]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [15]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Madhav")]})

AIMessage(content="Nice to meet you, Madhav. I'm happy to assist you with any questions or topics you'd like to discuss. How's your day going so far?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 59, 'total_tokens': 94, 'completion_time': 0.043830585, 'completion_tokens_details': None, 'prompt_time': 0.004253789, 'prompt_tokens_details': None, 'queue_time': 0.05004008, 'total_time': 0.048084374}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f84f9-91a6-7de2-bba2-757cc3593153-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 59, 'output_tokens': 35, 'total_tokens': 94})

In [16]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

/opt/anaconda3/envs/langchain/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [17]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Madhav")],
    config=config
)

response

AIMessage(content="Nice to meet you, Madhav. I'm a helpful assistant. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 59, 'total_tokens': 82, 'completion_time': 0.030059409, 'completion_tokens_details': None, 'prompt_time': 0.003928264, 'prompt_tokens_details': None, 'queue_time': 0.053876545, 'total_time': 0.033987673}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f84f9-eb39-7641-bc13-fffeae8df6ae-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 59, 'output_tokens': 23, 'total_tokens': 82})

In [18]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Madhav.'

In [19]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [20]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Madhav")],"language":"Hindi"})
response.content

'नमस्ते! मैं आपकी मदद करने के लिए यहाँ हूँ। मुझे खुशी है कि आप मेरा परिचय लेना चाहते हैं। आपका नाम माधव है, और मैं आपके साथ बात करने के लिए तैयार हूँ।'

In [21]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

/opt/anaconda3/envs/langchain/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [22]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Madhav")],"language":"Hindi"},
    config=config
)
repsonse.content

'नमस्ते माधव जी, मैं आपकी मदद करने के लिए तैयार हूँ। क्या मैं आपके लिए कुछ कर सकता हूँ?'

In [23]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

In [24]:
response.content


'माधव आपका नाम है।'